# SETU — SeqKD vs DPO-distill comparison (the paper's head-to-head)

Trains the **same** student two ways at matched size and evaluates both on the
**same real-reference dev set**:

- **S1 SeqKD** — SFT on the teacher's 1-best translations (Kim & Rush 2016 baseline).
- **S2 DPO-distill (ours)** — SFT on human references + DPO on ChrF-ranked preferences.

Set **GPU T4 x2** and **Internet On**. Uses `--limit 100000` (~5-6 h total for the
distill + two trainings); lower it if your GPU budget is tighter.

In [ ]:
import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!rm -rf /kaggle/working/SETU_v2
!git clone https://github.com/GeekyRiolu/SETU_v2.git /kaggle/working/SETU_v2
%cd /kaggle/working/SETU_v2/SETU
!pip -q install -e ".[data,teacher,prefs,quantize]"
!cp configs/model.gpu.yaml configs/model.yaml
!cp configs/training.gpu.yaml configs/training.yaml
!sed -i 's/device: cpu/device: cuda/' configs/teacher.yaml

In [ ]:
LIMIT = 100000
# data + preferences (preferences only needed for S2's DPO stage)
!setu-data --limit {LIMIT + 2000}
!setu-prefs --max-entries 8000

In [ ]:
# generate the teacher-distilled corpus (teacher 1-best targets) for SeqKD.
# This is the expensive step (teacher inference over LIMIT sources).
!setu-distill --limit {LIMIT} --batch-size 16
# verify it actually produced the corpus before training on it
import os
_dist = 'data/distilled/hin_Deva-eng_Latn/train.jsonl'
assert os.path.exists(_dist) and os.path.getsize(_dist) > 0, \
    'setu-distill did not produce a distilled corpus — check its output above'
print('distilled corpus rows:', sum(1 for _ in open(_dist)))

In [ ]:
# S1 SeqKD: SFT on teacher targets, no DPO. Eval is on real references.
# cp only runs if training succeeded (&&); a clear marker if it didn't.
!python scripts/train_full.py --train-corpus distilled --skip-dpo --limit {LIMIT} --dev-size 500 \
    && cp checkpoints/hin_Deva-eng_Latn/train_report.json /kaggle/working/report_S1_seqkd.json \
    && echo "=== S1 report saved ===" || echo "=== S1 TRAINING FAILED — see output above ==="

In [ ]:
# S0 SFT (human refs) + S2 SFT+DPO. Same size, same dev set.
!python scripts/train_full.py --train-corpus processed --limit {LIMIT} --dev-size 500 \
    && cp checkpoints/hin_Deva-eng_Latn/train_report.json /kaggle/working/report_S2_dpo.json \
    && echo "=== S2 report saved ===" || echo "=== S2 TRAINING FAILED — see output above ==="

In [ ]:
# comparison table (defensive — reports what's available)
import json, os

def load(p):
    return json.load(open(p)) if os.path.exists(p) else None

s1 = load('/kaggle/working/report_S1_seqkd.json')
s2 = load('/kaggle/working/report_S2_dpo.json')
if s1 is None:
    print('MISSING report_S1_seqkd.json — the S1 SeqKD cell did not finish (see its output).')
if s2 is None:
    print('MISSING report_S2_dpo.json — the S2 cell did not finish (see its output).')

def row(name, ev):
    r = ev.get('bleu_ratio')
    print(f"{name:26s} BLEU {ev['bleu']:6.2f}  chrF {ev['chrf']:6.2f}  ratio {r if r is None else round(r,3)}")

tb = (s2 or s1 or {}).get('sft_eval', {}).get('teacher_bleu')
print(f"\nteacher dev BLEU = {tb}\n")
if s1: row('S1 SeqKD (SFT-teacher)', s1['sft_eval'])
if s2: row('S0 SFT (human refs)', s2['sft_eval'])
if s2 and 'dpo_eval' in s2: row('S2 SFT-ref + DPO (ours)', s2['dpo_eval'])
print('\nPaste these into docs/PAPER_PLAN.md Table 1.')